<a href="https://colab.research.google.com/github/pras73/projects_repo/blob/main/Clinical_Decision_Support_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import torch
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
import cv2

# 1. Load Pretrained Deep Learning Vision Model (ResNet-50)
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.eval()

# 2. Define Image Preprocessing Pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Custom Grad-CAM Implementation for Model Interpretability
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

In [14]:
!pip install -q langchain-community chromadb langchain sentence-transformers langchain-core

import os
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Initialize open-source embeddings model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Synthetic Clinical Reference Database (Evidence-Based Guidelines)
clinical_guidelines = [
    Document(
        page_content="Acute Coronary Syndrome (ACS): Symptoms include chest pressure, radiation to left arm or jaw, dyspnea. Protocol: Obtain immediate 12-lead ECG, administer oxygen if SpO2 < 90%, initiate STAT cardiology consult.",
        metadata={"category": "Cardiology", "urgency": "High"}
    ),
    Document(
        page_content="Community-Acquired Pneumonia (CAP): Symptoms include fever, cough with sputum, pleuritic chest pain. Diagnosis confirmed via infiltrate on chest radiography. CURB-65 score guides admission decision.",
        metadata={"category": "Pulmonology", "urgency": "Medium"}
    ),
    Document(
        page_content="Acute Sepsis Protocol: Defined by suspected infection plus acute change in SOFA score. Red Flags: Systolic BP < 90 mmHg, lactate > 2 mmol/L, altered mental status. Action: Start broad-spectrum antibiotics within 1 hour.",
        metadata={"category": "Critical Care", "urgency": "Critical"}
    ),
    Document(
        page_content="Hypertensive Urgency vs Emergency: BP > 180/120 mmHg without end-organ damage is Urgency. Presence of end-organ damage (encephalopathy, acute kidney injury) constitutes Emergency requiring IV antihypertensives.",
        metadata={"category": "Emergency", "urgency": "High"}
    )
]

# Build Local Vector Index
vector_db = Chroma.from_documents(clinical_guidelines, embeddings, collection_name="clinical_kb")
print("✅ Clinical Knowledge Base indexed successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Clinical Knowledge Base indexed successfully!


In [15]:
from transformers import pipeline
from PIL import Image

# Load lightweight open-source vision model for image inspection
print("Loading Medical Vision Inspection Model...")
# Explicitly set the task to the supported 'image-text-to-text'
vision_analyzer = pipeline(task="image-text-to-text", model="Salesforce/blip-image-captioning-base")

def analyze_clinical_image(image: Image.Image) -> str:
    """Analyzes radiology scans or clinical photographs."""
    if image is None:
        return "No diagnostic image uploaded."

    results = vision_analyzer(image)
    caption = results[0]['generated_text']
    return f"Image Feature Analysis: Detected patterns consistent with '{caption}'."

Loading Medical Vision Inspection Model...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [16]:
def calculate_news_score(hr: int, sbp: int, spo2: int, temp: float) -> tuple[int, str]:
    """
    Computes a simplified National Early Warning Score (NEWS) based on vitals.
    Returns numeric score and risk category.
    """
    score = 0

    # Heart Rate Scoring
    if hr <= 40 or hr >= 131: score += 3
    elif hr >= 111 or hr <= 50: score += 2
    elif hr >= 91: score += 1

    # Systolic BP Scoring
    if sbp <= 90: score += 3
    elif sbp <= 100: score += 2
    elif sbp <= 110: score += 1

    # SpO2 Scoring
    if spo2 <= 91: score += 3
    elif spo2 <= 93: score += 2
    elif spo2 <= 95: score += 1

    # Risk Assessment
    if score >= 7 or sbp <= 90 or spo2 <= 91:
        return score, "CRITICAL / HIGH RISK (Immediate Emergency Escalation)"
    elif score >= 5:
        return score, "MODERATE RISK (Urgent Medical Evaluation Required)"
    else:
        return score, "LOW RISK (Standard Outpatient / Observation Routine)"

def clinical_agent_orchestrator(
    patient_id: str,
    symptoms: str,
    hr: int,
    sbp: int,
    spo2: int,
    temp: float,
    image: Image.Image = None
) -> dict:
    """
    Coordinates Vitals Evaluation, Guideline RAG Lookup, and Image Diagnostic Tools.
    """
    execution_trace = []

    # Step 1: Calculate Deterministic Risk Score
    score, risk_level = calculate_news_score(hr, sbp, spo2, temp)
    execution_trace.append(f"📊 [Vitals Engine]: Calculated NEWS Score = {score} ({risk_level})")

    # Step 2: Retrieve Relevant Clinical Protocols
    retrieved_docs = vector_db.similarity_search(symptoms, k=2)
    guidelines_text = "\n".join([f"• [{d.metadata['category']}] {d.page_content}" for d in retrieved_docs])
    execution_trace.append(f"📚 [Clinical RAG]: Matched {len(retrieved_docs)} evidence-based protocols.")

    # Step 3: Analyze Image (if provided)
    image_findings = analyze_clinical_image(image)
    if image is not None:
        execution_trace.append(f"👁️ [Vision Diagnostics]: {image_findings}")

    # Step 4: Red Flag Alert Detection
    red_flags = []
    if spo2 < 92: red_flags.append("Hypoxia (SpO2 < 92%)")
    if sbp < 90: red_flags.append("Hypotension (Systolic BP < 90 mmHg)")
    if "chest pain" in symptoms.lower(): red_flags.append("Possible Acute Coronary Syndrome")

    # Step 5: Synthesize Clinical Summary
    summary = []
    summary.append(f"## Clinical Assessment Report — Patient `{patient_id}`")
    summary.append(f"**Triage Level:** {risk_level}")

    if red_flags:
        summary.append(f"⚠️ **ACTIVE RED FLAGS DETECTED:** {', '.join(red_flags)}")

    summary.append(f"\n### Relevant Clinical Protocols:\n{guidelines_text}")

    if image is not None:
        summary.append(f"\n### Diagnostic Imaging Findings:\n{image_findings}")

    summary.append("\n*Disclaimer: This system is a clinical decision support tool prototype for educational demonstration and does not replace human clinical judgment.*")

    return {
        "report": "\n\n".join(summary),
        "trace": "\n".join(execution_trace)
    }

In [18]:
import gradio as gr

def ui_handler(patient_id, symptoms, hr, sbp, spo2, temp, image):
    if not symptoms.strip():
        return "Please describe patient symptoms.", "No action taken."

    result = clinical_agent_orchestrator(
        patient_id=patient_id if patient_id else "PT-000",
        symptoms=symptoms,
        hr=int(hr),
        sbp=int(sbp),
        spo2=int(spo2),
        temp=float(temp),
        image=image
    )
    return result["report"], result["trace"]

# Construct Dashboard
with gr.Blocks(theme=gr.themes.Soft(), title="Clinical Decision Support Agent") as demo:
    gr.Markdown("# 🩺 Multimodal Clinical Decision Support System (CDSS)")
    gr.Markdown("Real-time clinical triage assistant combining deterministic early warning scores (NEWS2), evidence-based RAG guidelines, and computer vision.")

    with gr.Row():
        with gr.Column(scale=1):
            patient_id = gr.Textbox(label="Patient ID", value="PT-8832")
            symptoms = gr.Textbox(label="Chief Complaint / Symptoms", placeholder="e.g., Severe chest pressure, shortness of breath, fever")

            gr.Markdown("#### Vital Signs")
            with gr.Row():
                hr = gr.Number(label="Heart Rate (BPM)", value=105)
                sbp = gr.Number(label="Systolic BP (mmHg)", value=88)
            with gr.Row():
                spo2 = gr.Number(label="SpO2 (%)", value=90)
                temp = gr.Number(label="Temp (°C)", value=38.5)

            image_input = gr.Image(type="pil", label="Upload Scan / X-Ray / Clinical Photo (Optional)")
            submit_btn = gr.Button("Run Clinical Assessment", variant="primary")

        with gr.Column(scale=1):
            report_output = gr.Markdown(label="Assessment Report")
            trace_output = gr.Code(label="Agent Decision Trace & Tool Calls", language=None)

    submit_btn.click(
        fn=ui_handler,
        inputs=[patient_id, symptoms, hr, sbp, spo2, temp, image_input],
        outputs=[report_output, trace_output]
    )

# Launch locally inside Colab cell without hanging public tunnels
demo.launch(inline=True)

/tmp/ipykernel_3371/3448984384.py:19: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Clinical Decision Support Agent") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b8a9b4c9252bd27f5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
